In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer
from datasets import load_dataset
from preprocessing import *

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
label_idx_to_name = {1: "World",  2: "Sports", 3: "Business", 4: "Sci/Tech"}

In [3]:
# Load the AG News dataset from Hugging Face
dataset = load_dataset("sh0416/ag_news")

# Convert to pandas DataFrames
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print("\nTrain data preview:")
print(train_df.head())

# Split into X (features) and y (labels)
X_train = train_df['description']
y_train = train_df['label']

X_test = test_df['description']
y_test = test_df['label']

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


Train set shape: (120000, 3)
Test set shape: (7600, 3)

Train data preview:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3      3  Iraq Halts Oil Exports from Main Southern Pipe...   
4      3  Oil prices soar to all-time record, posing new...   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  

X_train shape: (120000,)
y_train shape: (120000,)
X_test shape: (7600,)
y_test shape: (7600,)


In [18]:
X_train_preprocessed = text_normalization(X_train)
X_test_preprocessed = text_normalization(X_test)

In [ ]:
# Initialize CountVectorizer for binary bigram representation
bigram_vectorizer = CountVectorizer(
    ngram_range=(2, 2),  # Only bigrams (2-word sequences)
    binary=True,         # Binary representation (1 if present, 0 otherwise)
    token_pattern=r'\b\w+\b'  # Tokenize by word boundaries
)

# Fit on training data and transform both train and test
X_train_bigram = bigram_vectorizer.fit_transform(X_train_preprocessed)
X_test_bigram = bigram_vectorizer.transform(X_test_preprocessed)

print(f"Vocabulary size (unique bigrams): {len(bigram_vectorizer.vocabulary_)}")
print(f"X_train_bigram shape: {X_train_bigram.shape}")
print(f"X_test_bigram shape: {X_test_bigram.shape}")
print(f"Data type: {type(X_train_bigram)}")  # Sparse matrix for memory efficiency

# Show some example bigrams
print(f"\nFirst 10 bigrams in vocabulary:")
bigrams = list(bigram_vectorizer.vocabulary_.keys())[:10]
print(bigrams)

Vocabulary size (unique bigrams): 905473
X_train_bigram shape: (120000, 905473)
X_test_bigram shape: (7600, 905473)
Data type: <class 'scipy.sparse._csr.csr_matrix'>

First 10 bigrams in vocabulary:
['reuters short', 'short sellers', 'sellers wall', 'wall street', 'street s', 's dwindling', 'dwindling band', 'band of', 'of ultra', 'ultra cynics']


In [38]:
# decode the first row in X_train_bigram to see which bigrams are present
first_row_bigram = X_train_bigram[0].toarray()[0]
present_bigrams = [bigram for bigram, idx in bigram_vectorizer.vocabulary_.items() if first_row_bigram[idx] == 1]
print(f"\nBigrams present in the first training example:")
print(present_bigrams)
print(f"\nFirst training example (preprocessed): \n{X_train[0]}")


Bigrams present in the first training example:
['reuters short', 'short sellers', 'sellers wall', 'wall street', 'street s', 's dwindling', 'dwindling band', 'band of', 'of ultra', 'ultra cynics', 'cynics are', 'are seeing', 'seeing green', 'green again']

First training example (preprocessed): 
Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
